# FarmGuard – Animal Detection on Farmland (bounding boxes + parameters)

This notebook detects animals (and people / vehicles) in farm scenes, draws **bounding boxes**, and lets you tune and measure everything with **parameters**:

| Part | What it does |
|---|---|
| 1. Config | One cell with every parameter (model, thresholds, zones, voting, tracking) |
| 2. Single image | Boxes, confidence, threat level, "inside protected zone" check |
| 3. Parameter studies | Confidence sweep, input-size vs speed sweep |
| 4. Image folder | Batch detection with a CSV summary |
| 5. Video | Tracking, N-of-M voting, events, annotated video |
| 6. Evaluation | Precision / recall / F1 / AP50 on your own labeled images, and best threshold |
| 7. Optional | Fine-tune on your own data, export to ONNX |

**Three model modes** (set `mode` in the config):
- `open_vocab`: YOLO-World. You type the animal names (monkey, camel, donkey, ...) and it detects them **without training**. Handy for species COCO does not know. Accuracy varies, so always check it on your own images.
- `coco`: standard YOLO trained on COCO. More reliable, but it only knows dog, cat, horse, sheep, cow, elephant, bear, zebra, giraffe, bird, person and vehicles. **It cannot see monkeys, camels or donkeys.**
- `custom`: your own fine-tuned weights (`best.pt`) after you train on your data (section 7).

**Honest notes**
- Zero-shot results are a starting point, not a measured system. Use section 6 on your own labeled images before you trust any number.
- Ultralytics models use the AGPL-3.0 license. That is fine for an open-source project if you comply; check the terms before using them in a closed product.
- This notebook only *previews* the deterrent decision. It does not trigger any hardware.

In [ ]:
# 0) Install (run once). Colab already has most of these.
%pip install -q -U ultralytics opencv-python-headless pandas matplotlib

In [ ]:
# 1) Imports, configuration and threat table ---------------------------------------------
import os, sys, json, time, uuid, platform, shutil, subprocess, warnings
from dataclasses import dataclass, field, asdict, replace
from collections import Counter, deque
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

%matplotlib inline
warnings.filterwarnings("ignore")


@dataclass
class Config:
    # ---- Model -------------------------------------------------------------------------
    mode: str = "open_vocab"             # "open_vocab" | "coco" | "custom"
    weights: str = "yolov8s-worldv2.pt"  # open_vocab: YOLO-World file | coco: "yolov8s.pt" | custom: path to best.pt
    prompts: tuple = (                   # open_vocab only: the names you want it to find
        "monkey", "dog", "cat", "horse", "donkey", "camel", "cow", "wild boar",
        "deer", "bear", "bird", "person", "tractor", "car", "truck",
    )

    # ---- Inference ---------------------------------------------------------------------
    conf: float = 0.25                   # default confidence threshold
    iou: float = 0.50                    # NMS overlap threshold (lower = fewer overlapping boxes)
    imgsz: int = 640                     # input size (smaller = faster, larger = better for small animals)
    max_det: int = 50
    device: str = "auto"                 # "auto" | "cpu" | "cuda:0" | "mps"
    half: bool = True                    # half precision (only used on CUDA)

    # ---- Filtering ---------------------------------------------------------------------
    per_class_conf: dict = field(default_factory=lambda: {"cat": 0.40, "bird": 0.40, "person": 0.30})
    min_area_frac: float = 0.0005        # ignore boxes smaller than this fraction of the frame

    # ---- Zones: polygons with normalized (x, y) points between 0 and 1 ---------------------
    protected_zone: list = field(default_factory=lambda: [(0, 0), (1, 0), (1, 1), (0, 1)])  # whole frame
    ignore_zones: list = field(default_factory=list)  # e.g. [[(0.0, 0.0), (1.0, 0.0), (1.0, 0.15), (0.0, 0.15)]] for a road

    # ---- Video: tracking + N-of-M voting + events ------------------------------------------
    frame_stride: int = 2                # process every Nth frame
    track_iou: float = 0.30              # box overlap needed to keep the same track
    track_max_age: int = 8               # processed frames a track may be missing
    track_min_hits: int = 3              # frames before a track counts
    vote_n: int = 3                      # same label in at least N ...
    vote_m: int = 5                      # ... of the last M frames
    event_gap_s: float = 3.0             # seconds without the animal before an event ends
    record_vehicles: bool = False
    max_video_seconds: Optional[float] = None  # set e.g. 30 for a quick test

    # ---- Output ------------------------------------------------------------------------
    out_dir: str = "outputs"


cfg = Config()

# Threat table: label -> (threat level, what the system WOULD do). Edit for your region.
THREAT = {
    "monkey":    ("high",   "sound + strobe + alert"),
    "wild boar": ("high",   "sound + strobe + alert"),
    "boar":      ("high",   "sound + strobe + alert"),
    "deer":      ("high",   "sound + strobe + alert"),
    "nilgai":    ("high",   "sound + strobe + alert"),
    "cow":       ("medium", "sound + alert"),
    "cattle":    ("medium", "sound + alert"),
    "horse":     ("medium", "sound + alert"),
    "donkey":    ("medium", "sound + alert"),
    "camel":     ("medium", "sound + alert"),
    "sheep":     ("medium", "sound + alert"),
    "dog":       ("medium", "sound + alert"),
    "cat":       ("low",    "log only"),
    "bird":      ("low",    "log only"),
    "bear":      ("danger", "ALERT ONLY - keep away, no deterrent"),
    "elephant":  ("danger", "ALERT ONLY - keep away, no deterrent"),
}
# Things that are not animals. A person suppresses deterrents; vehicles are ignored.
NON_ANIMAL = {
    "person": ("person", "suppress deterrents"),
    "car": ("vehicle", "ignore"), "truck": ("vehicle", "ignore"), "bus": ("vehicle", "ignore"),
    "tractor": ("vehicle", "ignore"), "motorcycle": ("vehicle", "ignore"), "bicycle": ("vehicle", "ignore"),
}
# In coco mode only these labels are considered (the rest of the 80 classes are not farm-relevant).
COCO_RELEVANT = {"dog", "cat", "horse", "sheep", "cow", "elephant", "bear", "zebra", "giraffe", "bird",
                 "person", "car", "truck", "bus", "motorcycle", "bicycle"}

THREAT_COLORS = {  # BGR
    "high": (40, 40, 230), "medium": (0, 140, 255), "low": (230, 160, 40),
    "danger": (200, 40, 160), "person": (60, 200, 60), "vehicle": (150, 150, 150), "unknown": (200, 200, 0),
}
print("Config ready. Mode:", cfg.mode)

In [ ]:
# 2) Helpers: device, geometry, detections, filtering, inference, drawing --------------------
@dataclass
class Detection:
    label: str
    conf: float
    box: tuple                 # (x1, y1, x2, y2) in pixels
    area_frac: float = 0.0
    in_zone: bool = True
    threat: str = "unknown"
    action: str = ""
    status: str = "kept"       # kept | low_conf | too_small | outside_zone | ignore_zone | not_relevant
    track_id: Optional[int] = None


def resolve_device(pref="auto"):
    if pref != "auto":
        return pref
    try:
        import torch
        if torch.cuda.is_available():
            return "cuda:0"
        if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
            return "mps"
    except Exception:
        pass
    return "cpu"


def to_np(x):
    if hasattr(x, "cpu"):
        x = x.cpu()
    if hasattr(x, "numpy"):
        x = x.numpy()
    return np.asarray(x)


def iou_xyxy(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0


def to_px(poly_norm, w, h):
    return [(float(x) * w, float(y) * h) for x, y in poly_norm]


def point_in_poly(pt, poly_px):
    contour = np.array(poly_px, dtype=np.float32).reshape(-1, 1, 2)
    return cv2.pointPolygonTest(contour, (float(pt[0]), float(pt[1])), False) >= 0


def classify_threat(label):
    if label in NON_ANIMAL:
        return NON_ANIMAL[label]
    return THREAT.get(label, ("unknown", "alert only (unknown animal)"))


def relevant_labels(cfg, model):
    if cfg.mode == "coco":
        return set(COCO_RELEVANT)
    if cfg.mode == "open_vocab":
        return set(cfg.prompts)
    return None  # custom: everything the model knows


def filter_detections(dets, cfg, w, h, relevant=None):
    zone = to_px(cfg.protected_zone, w, h)
    ignores = [to_px(z, w, h) for z in cfg.ignore_zones]
    for d in dets:
        x1, y1, x2, y2 = d.box
        d.area_frac = max(0.0, (x2 - x1) * (y2 - y1)) / float(w * h)
        d.threat, d.action = classify_threat(d.label)
        foot = ((x1 + x2) / 2.0, y2)                       # bottom-center = where the animal stands
        d.in_zone = point_in_poly(foot, zone)
        thr = cfg.per_class_conf.get(d.label, cfg.conf)
        if relevant is not None and d.label not in relevant:
            d.status = "not_relevant"
        elif d.conf < thr:
            d.status = "low_conf"
        elif d.area_frac < cfg.min_area_frac:
            d.status = "too_small"
        elif any(point_in_poly(foot, z) for z in ignores):
            d.status = "ignore_zone"
        elif d.label not in NON_ANIMAL and not d.in_zone:  # people/vehicles count anywhere in view
            d.status = "outside_zone"
        else:
            d.status = "kept"
    return dets


def run_inference(model, img_bgr, cfg, device, relevant=None, conf_floor=None):
    """Returns (detections, speed_dict). conf_floor lowers the model threshold (used for sweeps / evaluation)."""
    h, w = img_bgr.shape[:2]
    floor = conf_floor if conf_floor is not None else min([cfg.conf] + list(cfg.per_class_conf.values()))
    use_half = bool(cfg.half and str(device).startswith("cuda"))
    r = model.predict(img_bgr, conf=float(floor), iou=cfg.iou, imgsz=cfg.imgsz, max_det=cfg.max_det,
                      device=device, half=use_half, verbose=False)[0]
    xyxy = to_np(r.boxes.xyxy).reshape(-1, 4)
    conf = to_np(r.boxes.conf).reshape(-1)
    cls = to_np(r.boxes.cls).reshape(-1).astype(int)
    names = r.names
    dets = [Detection(label=str(names[int(c)]), conf=float(s), box=tuple(float(v) for v in b))
            for b, s, c in zip(xyxy, conf, cls)]
    filter_detections(dets, cfg, w, h, relevant)
    return dets, dict(getattr(r, "speed", {}) or {})


def dets_to_df(dets, only_kept=True):
    rows = []
    for d in dets:
        if only_kept and d.status != "kept":
            continue
        x1, y1, x2, y2 = d.box
        rows.append(dict(label=d.label, conf=round(d.conf, 3), x1=round(x1), y1=round(y1), x2=round(x2), y2=round(y2),
                         area_frac=round(d.area_frac, 4), in_zone=d.in_zone, threat=d.threat, action=d.action,
                         status=d.status, track_id=d.track_id))
    cols = ["label", "conf", "x1", "y1", "x2", "y2", "area_frac", "in_zone", "threat", "action", "status", "track_id"]
    return pd.DataFrame(rows, columns=cols)


def draw(img_bgr, dets, cfg, show_rejected=False):
    out = img_bgr.copy()
    h, w = out.shape[:2]
    pts = np.array(to_px(cfg.protected_zone, w, h), dtype=np.int32)
    overlay = out.copy()
    cv2.fillPoly(overlay, [pts], (80, 200, 120))
    out = cv2.addWeighted(overlay, 0.10, out, 0.90, 0)
    cv2.polylines(out, [pts], True, (80, 200, 120), 2)
    for z in cfg.ignore_zones:
        cv2.polylines(out, [np.array(to_px(z, w, h), dtype=np.int32)], True, (60, 60, 230), 2)
    for d in dets:
        if d.status != "kept" and not show_rejected:
            continue
        color = THREAT_COLORS.get(d.threat, (200, 200, 0)) if d.status == "kept" else (150, 150, 150)
        x1, y1, x2, y2 = [int(v) for v in d.box]
        cv2.rectangle(out, (x1, y1), (x2, y2), color, 2)
        tag = f"{d.label} {d.conf:.2f}" + (f" #{d.track_id}" if d.track_id is not None else "")
        if d.status != "kept":
            tag += f" ({d.status})"
        (tw, th), _ = cv2.getTextSize(tag, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        cv2.rectangle(out, (x1, max(0, y1 - th - 6)), (x1 + tw + 4, y1), color, -1)
        cv2.putText(out, tag, (x1 + 2, max(th, y1 - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
    return out


def show(img_bgr, title="", figsize=(10, 6)):
    plt.figure(figsize=figsize)
    plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()


def load_image(path):
    img = cv2.imread(str(path))
    if img is None:
        raise FileNotFoundError(f"Could not read image: {path}. Put a farm photo there or change IMAGE_PATH. "
                                "In Colab: use the Files panel on the left to upload one.")
    return img


def save_run_config(cfg, out_dir):
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    info = {"time": time.strftime("%Y-%m-%d %H:%M:%S"), "python": sys.version.split()[0],
            "platform": platform.platform(), "config": asdict(cfg)}
    for mod in ("ultralytics", "torch", "cv2", "numpy"):
        try:
            info[mod] = __import__(mod).__version__
        except Exception:
            info[mod] = "n/a"
    path = Path(out_dir) / "run_config.json"
    path.write_text(json.dumps(info, indent=2, default=str))
    return path
print("Helpers ready.")

In [ ]:
# 3) Load the model --------------------------------------------------------------------------
def load_model(cfg):
    from ultralytics import YOLO
    if cfg.mode == "open_vocab":
        try:
            from ultralytics import YOLOWorld
            m = YOLOWorld(cfg.weights)
            m.set_classes(list(cfg.prompts))
            return m
        except Exception as e:
            print("Open-vocabulary mode failed:", repr(e))
            print("Common causes: no internet for the first download, or the text-encoder package could not install.")
            print("Falling back to COCO mode (it cannot detect monkey / camel / donkey).")
            cfg.mode, cfg.weights = "coco", "yolov8s.pt"
    return YOLO(cfg.weights)


device = resolve_device(cfg.device)
model = load_model(cfg)
relevant = relevant_labels(cfg, model)
Path(cfg.out_dir).mkdir(parents=True, exist_ok=True)
print("Mode:", cfg.mode, "| weights:", cfg.weights, "| device:", device)
print("Run config saved to:", save_run_config(cfg, cfg.out_dir))

## Section 1: One image

Put a farm photo at `IMAGE_PATH` (or change the path). Boxes are colored by threat level: red = high, orange = medium, blue = low, purple = danger (alert only), green = person, grey = vehicle. The light green shape is the protected zone.

In [ ]:
IMAGE_PATH = "samples/farm.jpg"      # <-- change me

img = load_image(IMAGE_PATH)
dets, speed = run_inference(model, img, cfg, device, relevant)

show(draw(img, dets, cfg), f"Detections ({sum(d.status == 'kept' for d in dets)} kept)")
print("Speed (ms):", {k: round(v, 1) for k, v in speed.items()})
display(dets_to_df(dets))

# Everything the filters rejected, and why (useful when something "should" have been found):
rejected = dets_to_df(dets, only_kept=False)
rejected = rejected[rejected["status"] != "kept"]
if len(rejected):
    print("Rejected detections:")
    display(rejected[["label", "conf", "area_frac", "in_zone", "status"]])

# Decision preview (no hardware is triggered):
kept = [d for d in dets if d.status == "kept"]
person_present = any(d.label == "person" for d in kept)
for d in kept:
    if d.threat in ("high", "medium"):
        verdict = "SUPPRESSED (person in view)" if person_present else d.action
    else:
        verdict = d.action
    print(f"  {d.label:<10} conf={d.conf:.2f}  threat={d.threat:<8} -> {verdict}")

## Section 2: Parameter studies

**Confidence sweep.** Run the model once at a very low threshold, then see how many detections each class keeps at every threshold. This shows where a class becomes stable and helps you pick `per_class_conf`.

**Input-size sweep.** Larger inputs find smaller or farther animals but run slower. The table shows the measured time for each size.

In [ ]:
# Confidence sweep (one inference, many thresholds) ----------------------------------------
raw, _ = run_inference(model, img, cfg, device, relevant, conf_floor=0.05)
raw = [d for d in raw if d.status != "not_relevant"]
thresholds = np.round(np.arange(0.05, 0.96, 0.05), 2)
labels_seen = sorted({d.label for d in raw})

if not raw:
    print("No detections even at 0.05. Try another image, a larger imgsz, or different prompts.")
else:
    plt.figure(figsize=(9, 4))
    for lab in labels_seen:
        counts = [sum(1 for d in raw if d.label == lab and d.conf >= t) for t in thresholds]
        plt.plot(thresholds, counts, marker="o", label=lab)
    plt.axvline(cfg.conf, color="k", linestyle="--", linewidth=1, label=f"current conf = {cfg.conf}")
    plt.xlabel("confidence threshold"); plt.ylabel("detections kept"); plt.title("Detections vs threshold")
    plt.legend(); plt.grid(alpha=0.3); plt.show()

In [ ]:
# Input-size sweep: speed and number of detections ---------------------------------------
rows = []
for size in (320, 416, 512, 640, 800):
    c2 = replace(cfg, imgsz=size)
    run_inference(model, img, c2, device, relevant)                 # warm-up
    times, n_kept = [], 0
    for _ in range(3):
        dets_s, sp = run_inference(model, img, c2, device, relevant)
        times.append(sum(sp.get(k, 0.0) for k in ("preprocess", "inference", "postprocess")))
        n_kept = sum(d.status == "kept" for d in dets_s)
    ms = float(np.median(times))
    rows.append(dict(imgsz=size, ms_per_image=round(ms, 1), fps=round(1000 / ms, 1) if ms else None, kept_detections=n_kept))
size_df = pd.DataFrame(rows)
display(size_df)
print("Device:", device, "| Note: this is YOUR machine's speed, not a Raspberry Pi's.")

## Section 3: A whole folder of images

Set `IMAGE_DIR`. The notebook saves annotated copies and a CSV of every kept detection, then shows counts per species. Use this on empty-field photos too: any detection there is a **false alarm**.

In [ ]:
IMAGE_DIR = "samples"                # <-- folder of images
SAVE_ANNOTATED = True

exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
files = sorted(p for p in Path(IMAGE_DIR).glob("*") if p.suffix.lower() in exts) if Path(IMAGE_DIR).exists() else []
if not files:
    print(f"No images found in '{IMAGE_DIR}'. Add some and re-run.")
else:
    out_img_dir = Path(cfg.out_dir) / "annotated"
    out_img_dir.mkdir(parents=True, exist_ok=True)
    all_rows, per_image = [], []
    for p in files:
        im = cv2.imread(str(p))
        if im is None:
            print("Skipping unreadable file:", p.name); continue
        dets_b, sp = run_inference(model, im, cfg, device, relevant)
        df_b = dets_to_df(dets_b); df_b.insert(0, "image", p.name)
        all_rows.append(df_b)
        per_image.append(dict(image=p.name, kept=len(df_b), ms=round(sum(sp.get(k, 0) for k in ("preprocess", "inference", "postprocess")), 1)))
        if SAVE_ANNOTATED:
            cv2.imwrite(str(out_img_dir / p.name), draw(im, dets_b, cfg))
    batch_df = pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()
    batch_df.to_csv(Path(cfg.out_dir) / "batch_detections.csv", index=False)
    display(pd.DataFrame(per_image))
    if len(batch_df):
        display(batch_df.groupby("label").agg(count=("label", "size"), mean_conf=("conf", "mean")).round(3))
    print("Saved:", Path(cfg.out_dir) / "batch_detections.csv")

## Section 4: Video with tracking, voting and events

Each frame is detected, boxes are linked into **tracks** (same animal over time), and a track only counts when the **same label appears in at least N of the last M frames**. That kills one-frame flickers. A confirmed track becomes an **event** with start/end time, species, confidence, and the decision the system *would* take.

The annotated video uses the `mp4v` codec. If your browser cannot play it, the notebook converts a copy with ffmpeg when ffmpeg is installed.

In [ ]:
class IoUTracker:
    """Tiny multi-object tracker: links boxes between frames by overlap (greedy)."""
    def __init__(self, iou_thr, max_age, vote_m):
        self.iou_thr, self.max_age, self.vote_m = iou_thr, max_age, vote_m
        self.tracks, self.next_id = {}, 1

    def update(self, dets, frame_idx):
        pairs = []
        for tid, t in self.tracks.items():
            for di, d in enumerate(dets):
                v = iou_xyxy(t["box"], d.box)
                if v >= self.iou_thr:
                    pairs.append((v, tid, di))
        pairs.sort(reverse=True)
        used_t, used_d = set(), set()
        for v, tid, di in pairs:
            if tid in used_t or di in used_d:
                continue
            used_t.add(tid); used_d.add(di)
            t, d = self.tracks[tid], dets[di]
            t.update(box=d.box, missed=0, last_frame=frame_idx)
            t["hits"] += 1
            t["history"].append((d.label, d.conf))
            d.track_id = tid
        for di, d in enumerate(dets):
            if di in used_d:
                continue
            tid = self.next_id; self.next_id += 1
            self.tracks[tid] = dict(box=d.box, hits=1, missed=0, first_frame=frame_idx, last_frame=frame_idx,
                                    history=deque([(d.label, d.conf)], maxlen=self.vote_m))
            d.track_id = tid
        for tid in list(self.tracks):
            if tid not in used_t and self.tracks[tid]["last_frame"] != frame_idx:
                self.tracks[tid]["missed"] += 1
                if self.tracks[tid]["missed"] > self.max_age:
                    del self.tracks[tid]


def vote_label(history, n):
    """Return the label that appears at least n times in the recent history, else None."""
    if not history:
        return None
    label, count = Counter(l for l, _ in history).most_common(1)[0]
    return label if count >= n else None


def finalize_event(ev, cfg):
    threat, action = classify_threat(ev["species"])
    kind = "person" if ev["species"] == "person" else ("vehicle" if threat == "vehicle" else "animal")
    would_deter = (kind == "animal" and threat in ("high", "medium") and not ev["person_seen"])
    return dict(event_id=ev["event_id"], track_id=ev["track_id"], kind=kind, species=ev["species"],
                start_s=round(ev["start_s"], 2), end_s=round(ev["end_s"], 2),
                duration_s=round(ev["end_s"] - ev["start_s"], 2), frames=ev["frames"],
                max_conf=round(max(ev["confs"]), 3), mean_conf=round(float(np.mean(ev["confs"])), 3),
                threat=threat, person_seen=ev["person_seen"], would_deter=would_deter,
                decision=("SUPPRESSED (person in view)" if (kind == "animal" and threat in ("high", "medium") and ev["person_seen"]) else action))


def to_h264(path):
    """Make a browser-playable copy if ffmpeg exists."""
    if shutil.which("ffmpeg") is None:
        return None
    dst = str(Path(path).with_name(Path(path).stem + "_h264.mp4"))
    r = subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(path), "-vcodec", "libx264", "-pix_fmt", "yuv420p", dst])
    return dst if r.returncode == 0 else None


def process_video(model, cfg, device, relevant, video_path, out_path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise FileNotFoundError(f"Could not open video: {video_path}")
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    writer = cv2.VideoWriter(str(out_path), cv2.VideoWriter_fourcc(*"mp4v"), max(1.0, fps / cfg.frame_stride), (W, H))
    if not writer.isOpened():
        print("Warning: could not open the video writer; continuing without an output video.")
        writer = None

    tracker = IoUTracker(cfg.track_iou, cfg.track_max_age, cfg.vote_m)
    open_events, done_events = {}, []
    idx, processed, t0 = -1, 0, time.perf_counter()
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        idx += 1
        t = idx / fps
        if cfg.max_video_seconds is not None and t > cfg.max_video_seconds:
            break
        if idx % cfg.frame_stride:
            continue
        dets, _ = run_inference(model, frame, cfg, device, relevant)
        kept = [d for d in dets if d.status == "kept"]
        tracker.update(kept, idx)
        person_now = any(d.label == "person" for d in kept)

        for d in kept:
            tr = tracker.tracks.get(d.track_id)
            if tr is None or tr["hits"] < cfg.track_min_hits:
                continue
            lab = vote_label(tr["history"], cfg.vote_n)
            if lab is None:
                continue
            if classify_threat(lab)[0] == "vehicle" and not cfg.record_vehicles:
                continue
            ev = open_events.get(d.track_id)
            if ev is None:
                ev = open_events[d.track_id] = dict(event_id=str(uuid.uuid4())[:8], track_id=d.track_id, species=lab,
                                                    start_s=t, end_s=t, confs=[], frames=0, person_seen=False)
            ev["species"], ev["end_s"] = lab, t
            ev["confs"].append(d.conf); ev["frames"] += 1
        if person_now:
            for ev in open_events.values():
                ev["person_seen"] = True
        for tid in list(open_events):
            if t - open_events[tid]["end_s"] > cfg.event_gap_s:
                done_events.append(finalize_event(open_events.pop(tid), cfg))

        if writer is not None:
            writer.write(draw(frame, dets, cfg))
        processed += 1
    cap.release()
    if writer is not None:
        writer.release()
    for ev in open_events.values():
        done_events.append(finalize_event(ev, cfg))
    wall = time.perf_counter() - t0
    stats = dict(frames_processed=processed, wall_seconds=round(wall, 2),
                 processing_fps=round(processed / wall, 2) if wall > 0 else None, video_fps=round(fps, 2))
    cols = ["event_id", "track_id", "kind", "species", "start_s", "end_s", "duration_s", "frames", "max_conf",
            "mean_conf", "threat", "person_seen", "would_deter", "decision"]
    return pd.DataFrame(done_events, columns=cols).sort_values("start_s").reset_index(drop=True), stats


def plot_events(events_df, title="Events timeline"):
    if events_df.empty:
        print("No events to plot."); return
    species = list(dict.fromkeys(events_df["species"]))
    plt.figure(figsize=(10, 0.6 * len(species) + 1.5))
    for i, sp in enumerate(species):
        for _, r in events_df[events_df["species"] == sp].iterrows():
            plt.barh(i, max(r["duration_s"], 0.2), left=r["start_s"], height=0.5)
    plt.yticks(range(len(species)), species); plt.xlabel("video time (s)"); plt.title(title)
    plt.grid(axis="x", alpha=0.3); plt.show()
print("Video tools ready.")

In [ ]:
VIDEO_PATH = "samples/farm.mp4"      # <-- change me
OUT_VIDEO = str(Path(cfg.out_dir) / "annotated.mp4")

if not Path(VIDEO_PATH).exists():
    print(f"Video not found: {VIDEO_PATH}. Set VIDEO_PATH to a clip (try cfg.max_video_seconds = 30 for a quick run).")
else:
    events_df, vstats = process_video(model, cfg, device, relevant, VIDEO_PATH, OUT_VIDEO)
    events_df.to_csv(Path(cfg.out_dir) / "events.csv", index=False)
    print("Processing stats:", vstats)
    display(events_df)
    plot_events(events_df)
    if Path(OUT_VIDEO).exists():
        playable = to_h264(OUT_VIDEO)
        print("Annotated video:", playable or OUT_VIDEO)
        if playable:
            from IPython.display import Video
            display(Video(playable, embed=False, width=720))

## Section 5: Measure accuracy on your own labeled images

You need **images** and **YOLO-format label files** (one `.txt` per image, each line: `class_id center_x center_y width height`, all normalized 0 to 1) plus a `classes.txt` (one class name per line, in order). Tools such as Label Studio, CVAT or Roboflow can export this format.

The notebook reports, per class: **AP50** (average precision at IoU 0.5), and **precision / recall / F1** at your current thresholds, plus the **best F1 threshold** for each class. Images with no label file count as **empty scenes**, so every detection there is a false positive. Keep a separate test set that comes from a different place than any data you train on.

If the model's names differ from your class names, edit `LABEL_MAP` (for example `{"cow": "cattle"}`).

In [ ]:
def load_yolo_labels(txt_path, w, h, class_names):
    gts = []
    if not Path(txt_path).exists():
        return gts
    for line in Path(txt_path).read_text().splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        c = int(float(parts[0])); cx, cy, bw, bh = [float(v) for v in parts[1:5]]
        gts.append((class_names[c], ((cx - bw / 2) * w, (cy - bh / 2) * h, (cx + bw / 2) * w, (cy + bh / 2) * h)))
    return gts


def match_image(preds, gts, iou_thr):
    """preds: [(label, conf, box)], gts: [(label, box)] -> list of (label, conf, is_tp)."""
    used, out = set(), []
    for label, conf, box in sorted(preds, key=lambda p: -p[1]):
        best, best_iou = -1, iou_thr - 1e-9
        for gi, (glabel, gbox) in enumerate(gts):
            if gi in used or glabel != label:
                continue
            v = iou_xyxy(box, gbox)
            if v > best_iou:
                best, best_iou = gi, v
        if best >= 0:
            used.add(best)
        out.append((label, conf, best >= 0))
    return out


def average_precision(tp, conf, n_gt):
    if n_gt == 0 or len(tp) == 0:
        return float("nan")
    order = np.argsort(-np.asarray(conf))
    tp = np.asarray(tp, dtype=float)[order]
    fp = 1.0 - tp
    tpc, fpc = np.cumsum(tp), np.cumsum(fp)
    recall = tpc / (n_gt + 1e-9)
    precision = tpc / (tpc + fpc + 1e-9)
    mrec = np.concatenate(([0.0], recall, [1.0]))
    mpre = np.concatenate(([1.0], precision, [0.0]))
    mpre = np.flip(np.maximum.accumulate(np.flip(mpre)))
    idx = np.where(mrec[1:] != mrec[:-1])[0]
    return float(np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1]))


def evaluate_folder(model, cfg, device, img_dir, label_dir, class_names, label_map=None, iou_thr=0.5, floor=0.05):
    label_map = label_map or {}
    files = sorted(p for p in Path(img_dir).glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"})
    records = []                               # (label, conf, is_tp)
    n_gt = Counter()
    n_images = 0
    for p in files:
        im = cv2.imread(str(p))
        if im is None:
            continue
        n_images += 1
        h, w = im.shape[:2]
        gts = load_yolo_labels(Path(label_dir) / (p.stem + ".txt"), w, h, class_names)
        for g in gts:
            n_gt[g[0]] += 1
        dets, _ = run_inference(model, im, cfg, device, relevant=None, conf_floor=floor)
        preds = []
        for d in dets:
            lab = label_map.get(d.label, d.label)
            if lab in class_names and d.area_frac >= cfg.min_area_frac:
                preds.append((lab, d.conf, d.box))
        records.extend(match_image(preds, gts, iou_thr))

    rows, curves = [], {}
    for lab in class_names:
        recs = [(c, t) for (l, c, t) in records if l == lab]
        conf = np.array([c for c, _ in recs]); tp = np.array([t for _, t in recs], dtype=float)
        ng = n_gt[lab]
        thr = cfg.per_class_conf.get(lab, cfg.conf)
        tp_at = float(np.sum(tp[conf >= thr])) if len(conf) else 0.0
        fp_at = float(np.sum(1 - tp[conf >= thr])) if len(conf) else 0.0
        prec = tp_at / (tp_at + fp_at) if (tp_at + fp_at) > 0 else float("nan")
        rec = tp_at / ng if ng else float("nan")
        f1 = 2 * prec * rec / (prec + rec) if (prec == prec and rec == rec and (prec + rec) > 0) else float("nan")
        best_thr, best_f1 = float("nan"), float("nan")
        if len(conf) and ng:
            order = np.argsort(-conf)
            c_s, t_s = conf[order], tp[order]
            tpc, fpc = np.cumsum(t_s), np.cumsum(1 - t_s)
            p_c, r_c = tpc / (tpc + fpc + 1e-9), tpc / ng
            f1_c = 2 * p_c * r_c / (p_c + r_c + 1e-9)
            k = int(np.argmax(f1_c)); best_thr, best_f1 = float(c_s[k]), float(f1_c[k])
            curves[lab] = (r_c, p_c)
        rows.append(dict(cls=lab, n_gt=ng, n_pred=len(conf), AP50=average_precision(tp, conf, ng),
                         precision=prec, recall=rec, F1=f1, op_conf=thr, best_F1=best_f1, best_conf=best_thr))
    df = pd.DataFrame(rows)
    # empty-scene false positives at the operating thresholds
    fp_total = sum(1 for (l, c, t) in records if (not t) and c >= cfg.per_class_conf.get(l, cfg.conf))
    return df, curves, dict(images=n_images, false_positives_at_op=fp_total,
                            fp_per_image=round(fp_total / n_images, 3) if n_images else None)
print("Evaluation tools ready.")

In [ ]:
EVAL_IMG_DIR   = "eval/images"        # <-- your test images
EVAL_LABEL_DIR = "eval/labels"        # <-- YOLO .txt files
CLASSES_FILE   = "eval/classes.txt"   # <-- one class name per line
LABEL_MAP      = {}                   # e.g. {"cow": "cattle"}

if not (Path(EVAL_IMG_DIR).exists() and Path(CLASSES_FILE).exists()):
    print("Evaluation data not found. Create eval/images, eval/labels and eval/classes.txt, then re-run.")
else:
    class_names = [l.strip() for l in Path(CLASSES_FILE).read_text().splitlines() if l.strip()]
    eval_df, curves, extra = evaluate_folder(model, cfg, device, EVAL_IMG_DIR, EVAL_LABEL_DIR, class_names, LABEL_MAP)
    display(eval_df.round(3))
    print("Overall:", extra, "| mean AP50:", round(float(np.nanmean(eval_df["AP50"])), 3))
    eval_df.to_csv(Path(cfg.out_dir) / "evaluation.csv", index=False)
    if curves:
        plt.figure(figsize=(6, 5))
        for lab, (r_c, p_c) in curves.items():
            plt.plot(r_c, p_c, label=lab)
        plt.xlabel("recall"); plt.ylabel("precision"); plt.title("Precision-recall (IoU 0.5)")
        plt.legend(); plt.grid(alpha=0.3); plt.show()
    print("Tip: copy the best_conf values into cfg.per_class_conf, then re-run to see the effect.")

## Section 6 (optional): Fine-tune on your own data

Turn this on only when you have a labeled dataset. Rules that keep results honest:
- **Split by source or location**, never by random frames. Otherwise near-identical frames leak into the test set and the scores look better than they are.
- Check the **license** of every image you use, and keep a manifest.
- Use the training set for training, the validation set to choose settings, and the test set only once at the end.

Expected folder layout (a standard YOLO dataset):
```
dataset/
  images/train  images/val  images/test
  labels/train  labels/val  labels/test
```

In [ ]:
RUN_TRAINING = False                 # <-- set True to train
DATASET_ROOT = "dataset"
TRAIN_CLASSES = ["monkey", "dog", "cat", "horse", "donkey", "camel", "cow", "wild boar", "deer", "person", "vehicle"]
BASE_WEIGHTS = "yolov8n.pt"          # small and fast; try yolov8s.pt for more accuracy
EPOCHS, BATCH = 50, 16

def write_dataset_yaml(root, names, path="farm_data.yaml"):
    import yaml
    d = {"path": str(Path(root).resolve()), "train": "images/train", "val": "images/val", "test": "images/test",
         "names": {i: n for i, n in enumerate(names)}}
    Path(path).write_text(yaml.safe_dump(d, sort_keys=False))
    return path

if RUN_TRAINING:
    from ultralytics import YOLO
    data_yaml = write_dataset_yaml(DATASET_ROOT, TRAIN_CLASSES)
    trainer = YOLO(BASE_WEIGHTS)
    trainer.train(data=data_yaml, epochs=EPOCHS, imgsz=cfg.imgsz, batch=BATCH, device=device,
                  project=str(Path(cfg.out_dir) / "train"), name="farmguard", patience=15)
    best = Path(cfg.out_dir) / "train" / "farmguard" / "weights" / "best.pt"
    print("Best weights:", best, "| exists:", best.exists())
    print("To use them: cfg.mode = 'custom'; cfg.weights = str(best); then re-run the model-loading cell.")
else:
    print("Training skipped (RUN_TRAINING = False).")

## Section 7 (optional): Export to ONNX

An ONNX file can run in ONNX Runtime (for example inside the FarmGuard edge program) without PyTorch. Re-check speed and accuracy after exporting, because the exported model can differ slightly.

In [ ]:
RUN_EXPORT = False                   # <-- set True to export
if RUN_EXPORT:
    exported = model.export(format="onnx", imgsz=cfg.imgsz, simplify=True)
    print("Exported:", exported)
    try:
        import onnxruntime as ort
        sess = ort.InferenceSession(str(exported), providers=["CPUExecutionProvider"])
        print("ONNX inputs:", [(i.name, i.shape) for i in sess.get_inputs()])
        print("ONNX outputs:", [(o.name, o.shape) for o in sess.get_outputs()])
    except Exception as e:
        print("onnxruntime check skipped:", repr(e))
else:
    print("Export skipped (RUN_EXPORT = False).")

## What to do next

1. Run it on your own photos and clips. Look at the **rejected detections** table to see why something was missed.
2. Label 100 to 300 test images from a place the model has not seen, then run Section 5 and set `per_class_conf` from the best-F1 thresholds.
3. If `open_vocab` is not accurate enough for a species, collect and label data and fine-tune (Section 6), then compare the AP50 numbers.
4. Record only measured numbers. Do not claim field results from this notebook.
5. Move the chosen model into the FarmGuard edge program (Prompt 3 in the PC Edition plan).

**Limitations:** boxes depend heavily on lighting, distance and background. Night and infrared images need their own test data. Zero-shot prompts can confuse similar animals (donkey vs horse, cat vs small dog). Always report results per class and per condition.